# Species migration portfolio assignment

## Step 1: Get occurrence data from the Global Biodiversity Information Facility (GBIF)
For your species migration portfolio assignment, you will need to acquire GBIF data for a different species. I recommend choosing a terrestrial species, as this will work best with the ecoregions data used in the assignment.

## Set up
In the imports cell, we’ve included some packages that you will need. Add imports for additional packages that will help you to:
1. Work with reproducible file paths
2. Work with tabular data

In [3]:
import time ### time processing steps
import zipfile ### read and extract .zip folders
from getpass import getpass ### handles sensitive inputs like passwords and API tokens
from glob import glob ### function for finding file paths
import requests ### make requests to websites and web APIs

### GBIF packages
import pygbif.occurrences as occ ### get GBIF occurrence data
import pygbif.species as species ### deals with species names

### import additional packages you'll need
### reproducible file paths
import os
import pathlib
import shutil

### tabular and spatial data
import pandas as pd
import geopandas as gpd

### libraries for dynamic mapping
import cartopy.crs as ccrs
import holoviews as hv
import panel as pn
import hvplot.pandas

### get month names
import calendar

### activate panel and hvplot extensions for the interactive plot
pn.extension()
hv.extension("bokeh")

ImportError: DLL load failed while importing _transformer: 애플리케이션 제어 정책에서 이 파일을 차단했습니다.

### Step 1A: Make a GBIF account
You will need to make a free account with [GBIF](https://www.gbif.org/) to access their data. Keep track of your username, password, and the email address you sign up with, as you'll need all of this later!

You will need a (free) [GBIF account](https://www.gbif.org/) to complete this challenge. Then, run the following code to enter your credentials for the rest of your session.

This code is **interactive**, meaning that it will **ask you for a response**! The prompt can sometimes be
hard to see if you are using VSCode – it appears at the **top** of your editor window.

**Tips:**
1. If you need to save credentials across multiple sessions, you can consider loading them in from a file like a `.env`, but make sure to add it to .gitignore so you don’t commit your credentials to your repository!
2. If you accidentally enter your credentials wrong, you can set `reset = True` instead of `reset = False`.

**Warning**
Your email address **must** match the email you used to sign up for GBIF!

In [ ]:
### change this to reset = False after you run this chunk the first time
reset = False

####--------------------------------------####
#### DO NOT MODIFY THE REST OF THIS CODE! ####
####--------------------------------------####
### This code asks for your credentials and saves it for the rest of the session.
### NEVER put your credentials into your code!!!!
### GBIF needs a username, password, and email 

### request and store username
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

### securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
### request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

### Step 1B: Create a project folder and data directories
The code below will help you get started with making a folder for your data to live in:

In [ ]:
### create data directory
data_dir = os.path.join(

    ### point to your home directory
    pathlib.Path.home(),

    ### nest the data in the folder where your assignments live
    'earth-analytics',
    'data',
    'crane-migration')

### make the directory
os.makedirs(data_dir, exist_ok = True)

### define directory for the gbif data
gbif_dir = os.path.join(data_dir, 'gbif')

### make the directory
os.makedirs(gbif_dir, exist_ok = True)

### STEP 1C: Get the taxon key from GBIF

One of the tricky parts about getting occurrence data from GBIF is that species often have multiple names in different contexts. Luckily, GBIF also provides a Name Backbone service that will translate scientific and colloquial names into unique identifiers. GBIF calls these identifiers **taxon keys**:

1. Put the species name into the correct location in the code below.
2. Examine the object you get back from the species query. What part of it do you think might be the taxon key
3. Extract and save the taxon key.

In [ ]:
### define the species name you will be using
backbone = species.name_backbone(scientificName = 'Antigone vipio')

### check it out
backbone

{'usage': {'key': '9473367',
  'name': 'Antigone vipio (Pallas, 1811)',
  'canonicalName': 'Antigone vipio',
  'authorship': '(Pallas, 1811)',
  'rank': 'SPECIES',
  'code': 'ZOOLOGICAL',
  'status': 'SYNONYM',
  'genericName': 'Antigone',
  'specificEpithet': 'vipio',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Antigone</i> <i>vipio</i> (Pallas, 1811)'},
 'acceptedUsage': {'key': '2474942',
  'name': 'Grus vipio Pallas, 1811',
  'canonicalName': 'Grus vipio',
  'authorship': 'Pallas, 1811',
  'rank': 'SPECIES',
  'code': 'ZOOLOGICAL',
  'genericName': 'Grus',
  'specificEpithet': 'vipio',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Grus</i> <i>vipio</i> Pallas, 1811'},
 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'},
  {'key': '44', 'name': 'Chordata', 'rank': 'PHYLUM'},
  {'key': '212', 'name': 'Aves', 'rank': 'CLASS'},
  {'key': '1493', 'name': 'Gruiformes', 'rank': 'ORDER'},
  {'key': '9313', 'name': 'Gruidae', 'rank': 'FAMILY'},
  {'key': '2474940', '

Depending on the output from "backbone," you may need to modify the column in the code below to use 'usageKey' or another column:

In [ ]:
### pull out the species key
species_key = backbone['acceptedUsage']['key']
species_key

'2474942'

### Step 1D: Download data from GBIF

Downloading GBIF data is a multi-step process. However, we’ve provided you with a chunk of code that handles the API communications and caches the download. You’ll still need to customize your search.

Submit a request to GBIF using the following steps:
1. Replace `csv_file_pattern` with a string that will match **any** `.csv` file when used in the `.rglob()` method. HINT: The character `*` represents any number of any values except the file separator (e.g. `/` on UNIX systems).
2. Add parameters to the GBIF download function, `occ.download()` to limit your query to:
    - observations of 
    - from 
    - with spatial coordinates.
3. Then, run the download. This can take a few minutes. You can check your downloads by logging on to the [GBIF website](https://www.gbif.org/user/download").

In [ ]:
### set file name for download
gbif_pattern = os.path.join(gbif_dir, '*.csv')

### set the code to only download your data the first time you run this chunk (since it can be slow)
if not glob(gbif_pattern):

    ### only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:

        ### submit a request to GBIF
        gbif_query = occ.download([

            ### add your species key
            f"speciesKey = {species_key}",

            ### filter out results without geographic coordinates
            'hasCoordinate = True',

            ### choose a year to include
            f'year = 2024',
        ])

        ### grab the first result
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    ### wait for the download to build
    download_key = os.environ['GBIF_DOWNLOAD_KEY']

    ### use the occurrence command module in pygbif to get the metadata
    wait = occ.download_meta(download_key)['status']

    ### check if the status of the download = "SUCCEEDED"
    while not wait=='SUCCEEDED':
        wait = occ.download_meta(download_key)['status']

        ### wait before re-querying the API
        time.sleep(5)

    ### download GBIF data when it's ready
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'], 
        path = gbif_dir)

    ### unzip GBIF data using the zipfile package
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

### find the extracted .csv file path (take the first result)
gbif_path = glob(gbif_pattern)[0]

You might notice that the GBIF data filename isn’t very **descriptive**. At this point, you may want to clean up your data directory so that you know what the file is later on!

1. Replace ‘your-gbif-filename’ with a descriptive name.
2. Run the cell.
3. Check your data folder. Is it organized the way you want?

Tip: you can use the `shutil` package to programmatically rename the file.

In [ ]:
### give the download a descriptive name
new_gbif_path = os.path.join(gbif_dir, 'white-naped-crane-gbif-2024.csv')

### move file to descriptive path (only if it hasn't been renamed yet)
if gbif_path != new_gbif_path:
    shutil.move(gbif_path, new_gbif_path)

### use the renamed file from here on
gbif_path = new_gbif_path

### Step 1E: Load the GBIF data into Python

Just like you did when wrangling your data from the data subset, you’ll need to load your GBIF data and convert it to a GeoDataFrame.

In [ ]:
### load the GBIF data (GBIF downloads are tab-delimited)
gbif_df = pd.read_csv(
    gbif_path,
    delimiter = '\t',
    index_col = 'gbifID',
    ### keep only the ID, location, and month columns needed for mapping
    usecols = ['gbifID', 'decimalLatitude', 'decimalLongitude', 'month']
)

### convert to GeoDataFrame
gbif_gdf = (
    gpd.GeoDataFrame(
        gbif_df,

        ### set geometry from the lon/lat of each GBIF record
        geometry = gpd.points_from_xy(
            gbif_df.decimalLongitude,
            gbif_df.decimalLatitude),

        ### set the coordinate reference system
        crs = "EPSG:4326")

    ### select the desired columns
    [['month', 'geometry']]
)

### check results
gbif_gdf.total_bounds

array([102.64507 ,  28.757471, 145.57321 ,  51.234487])

## Wrap up Portfolio Step 1

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!

## Step 2: Map the migration of White-naped Cranes

In [ ]:
### get url for ecoregions
ecoreg_url = ("https://storage.googleapis.com/"
       "teow2016/Ecoregions2017.zip")

### name the ecoregions directory and file
ecoregions_dir = os.path.join(data_dir, 'resolve_ecoregions')
ecoregions_path = os.path.join(ecoregions_dir, 'ecoregions.shp')

### make the ecoregions directory
os.makedirs(ecoregions_dir, exist_ok = True)

### download the data once
if not os.path.exists(ecoregions_path):
    print('Downloading ecoregions...')
    ecoregions_download = gpd.read_file(ecoreg_url)
    ecoregions_download.to_file(ecoregions_path)
else:
    print('Ecoregions already downloaded:', ecoregions_path)

### open up the ecoregions boundaries
ecoregions_gdf = gpd.read_file(ecoregions_path)

INFO:Created 847 records


In [ ]:
### find the ecoregion for each observation
gbif_ecoregion_gdf = (
    ecoregions_gdf
    .to_crs(gbif_gdf.crs)
    .sjoin(gbif_gdf, how = 'inner', predicate = 'contains')
    .rename(columns = {'OBJECTID': 'ecoregion'})
)

### count the observations in each ecoregion each month
occurrence_df = (
    gbif_ecoregion_gdf
    .groupby(['ecoregion', 'month'])
    .agg(occurrences = ('gbifID', 'count'))
)

### get rid of rare observations (which are possibly misidentifications)
occurrence_df = occurrence_df[occurrence_df.occurrences > 1]

### calculate the mean number of observations by ecoregion and by month
mean_occurrences_by_ecoregion = occurrence_df.groupby(['ecoregion']).mean()
mean_occurrences_by_month = occurrence_df.groupby(['month']).mean()

### normalize by space and time for sampling effort
occurrence_df['norm_occurrences'] = (
    occurrence_df
    / mean_occurrences_by_ecoregion
    / mean_occurrences_by_month
)

### check it out
occurrence_df

occurrences  norm_occurrences
ecoregion month                               
25.0      4                5          0.146520
          5                7          0.026774
          6                7          0.027100
          9                2          0.032653
75.0      11               3          0.077670
136.0     3               10          0.133809
          11               6          0.073582
          12               3          0.014416
142.0     1               80          0.066582
          2               40          0.038586
          3               19          0.045266
          4                3          0.012975
          10              10          0.031236
          11              46          0.100441
          12              51          0.043635
156.0     1               58          0.033021
          2               15          0.009898
          11              11          0.016430
          12             124          0.072575
200.0     4                8          0.027555
          5              111          0.049901
          6              117          0.053240
          7                2          0.001690
          9               22          0.042217
          10               8          0.019900
322.0     1               47          0.035908
          2               87          0.077040
          3                7          0.015309
          12              14          0.010996
325.0     3                9          0.108959
          11               5          0.055479
354.0     1                3          0.029605
416.0     1               19          0.048491
          2                9          0.026623
          3               14          0.102279
          11               3          0.020087
          12              13          0.034108
450.0     4               10          0.029777
          5               96          0.037311
          6              110          0.043273
          7               51          0.037249
          8               32          0.019355
          9               11          0.018249
481.0     1                2          0.029605
620.0     5               16          0.029208
          6                6          0.011086
682.0     1               46          0.056158
          2                9          0.012735
          11              27          0.086478
          12              15          0.018826
703.0     5               19          0.030522
          6                6          0.009756
719.0     1               41          0.057801
          2               31          0.050654
          11               2          0.007397
          12              10          0.014493
808.0     1                8          0.022556
          2               13          0.042484

In [ ]:
### simplify the geometry to speed up plotting
### (preserve_topology = False avoids distorted polygons on the map)
ecoregions_gdf.geometry = ecoregions_gdf.simplify(
    .05, preserve_topology = False)

### remove tiny ecoregions that became empty after simplifying
ecoregions_gdf = ecoregions_gdf[ecoregions_gdf.is_empty == False]

### change the CRS to Mercator for mapping
ecoregions_gdf = ecoregions_gdf.to_crs(ccrs.Mercator())

### make ecoregion ID the index to match occurrence_df
ecoregions_gdf = (
    ecoregions_gdf
    .rename(columns = {'OBJECTID': 'ecoregion'})
    .set_index('ecoregion')
)

### join the occurrences with the ecoregion shapes
occurrence_gdf = ecoregions_gdf.join(occurrence_df)

In [ ]:
### make the basemap tile object
carto_tiles = hv.Tiles(
    ("https://basemaps.cartocdn.com/rastertiles/voyager/{Z}/{X}/{Y}.png"
     "?key=cb1_446c_1_8946b413de0ace50f732dbf1"),
    name = "CARTO"
)

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month',
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        c = 'norm_occurrences',
        groupby = 'month',
        geo = True, crs = ccrs.Mercator(),

        ### add the basemap
        tiles = carto_tiles,

        ### add a title
        title = "White-naped Crane observations by ecoregion and month (2024)",

        ### set color map and label the color bar
        cmap = 'viridis',
        clabel = 'Normalized occurrences',

        ### set plot boundaries and size
        xlim = (xmin, xmax), ylim = (ymin, ymax),
        frame_height = 600, frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = True,
        widgets = {'month': slider}
    )
)

### check it out
pn.panel(migration_plot)

BokehModel(combine_events=True, render_bundle={'docs_json': {'e81cc87d-7254-479d-997e-d0403002741c': {'version…

In [ ]:
### save the map in the img folder of the portfolio repository
### (the notebook runs from the notebooks folder, so go up one level)
map_path = os.path.join(os.pardir, 'img', 'white-naped-crane-migration.html')

### export the map with the basemap as an .html file
migration_plot.save(map_path, embed = True, resources = "inline")